# Full Trenberth Diagram Calibration

This notebook calibrates **20 parameters** spanning shortwave radiation, longwave radiation, and surface albedo to simultaneously match the [Trenberth et al. (2009)](https://doi.org/10.1175/2008BAMS2634.1) global energy budget across **6 fluxes**:

| Flux | Target | Description |
|------|--------|-------------|
| OSR  | 101.9 W/m² | Outgoing shortwave at top-of-atmosphere |
| SRU  | 23.0 W/m²  | Surface shortwave reflected upward |
| SRD  | 168.0 W/m² | Surface shortwave absorbed downward |
| OLR  | 235.0 W/m² | Outgoing longwave at top-of-atmosphere |
| LRD  | 333.0 W/m² | Surface longwave downward (greenhouse back-radiation) |
| LRU  | 398.0 W/m² | Surface longwave upward (Stefan-Boltzmann emission) |

This is an extension of `radiation_sw.ipynb`: we add longwave emissivities and transmissivity parameters. Including OLR in the loss couples the SW and LW optimisation — changing cloud albedo now also feeds back through the LW budget via surface temperature.

**Fix (2026-07-30):** this notebook originally trained 23 parameters (including 3 land-hydrology params) and diverged. Root-caused and fixed — see the note in Section 2's parameter-definition cell and `project_trenberth_lw_transmissivity_gradscale_fix` memory for full detail. Two changes: (1) `grad_scale` added to the Frierson LW-transmissivity/emissivity block, which was fighting the SW-albedo block for a shared `grad_clip` budget; (2) 3 land-hydrology parameters removed after confirming bit-exact-zero AD gradient in every run.

**Update (2026-07-31):** the grad_scale fix alone stopped the catastrophic blowup but the run still didn't converge. Root-caused (see Section 5's status note): `lrd`'s bias was drifting monotonically from batch 1 the whole time, just masked early by `srd`'s larger, faster-shrinking initial bias. A `batch_days` sensitivity sweep (30/10/5/3/2) found shorter windows are strictly better on every metric — `batch_days=2` reaches a true floor of `best_smoothed_loss=148.8` (vs. 889.4 at the original `batch_days=10`) with 5 of 6 fluxes essentially nailed, `lrd` the one remaining gap.

**Update (2026-08-01):** closed most of that remaining `lrd` gap via a corrected loss-reweighting sweep — see Section 3's markdown for why the first reweighting attempt looked like a dead end but wasn't (comparison-methodology bug, corrected). Final production config (Section 5): `batch_days=2` + `lrd` weight=**0.7**, validated to hold at true 7-year climate equilibrium (31% better than the unweighted config, not just a training-metric artifact). Full derivation in `project_trenberth_lw_transmissivity_gradscale_fix` memory.

**Expected runtime:** ~1 hour for training (Section 5) + ~1 hour for climate validation (Section 8) at T31 — both much faster than the original ~8-12h / ~1-3h estimates now that `batch_days=2` is adopted, and Section 5 is resume-safe (loads the completed result instantly if `output/trenberth_full_result.jld2` already exists).

## 1. Setup

In [ ]:
using Pkg
Pkg.activate(joinpath(@__DIR__, ".."))

using SpeedyCalibration
using Optimisers
using CairoMakie
using GeoMakie
using Dates
using Printf

  Activating project at `~/master_thesis/Code_SpeedyWeather/SpeedyCalibration.jl`
[ Info: Precompiling SpeedyCalibration [5d65bc14-e915-412c-9a7c-d2e552044b02]
[ Info: Precompiling CairoMakie [13f3f980-e62b-5c42-98c6-ff1f3baf88f0]


## 2. Define Trainable Parameters

We train **20 parameters** across three modules (shortwave, longwave, surface albedo). All paths and bounds are copied
directly from `full_trend_birth_training.ipynb`, with `grad_scale` corrections described below (2026-07-30 fix).

**`absorptivity_water_vapor` needs `grad_scale=0.01`.**
The physical gradient is ~200× weaker than the cloud parameters because it is
multiplied by specific humidity (q ≈ 0.005). Its lower bound is 60 (not 0) because
the model becomes numerically unstable below ~57.

**`tau0_equator`/`tau0_pole`/`fl` (Frierson LW transmissivity) need `grad_scale=(0.04, 0.25, 0.025)`.**
Unlike `absorptivity_water_vapor`, these never received a `grad_scale` when added, and their raw
AD gradients (~400/~60/~600, measured in the 6-stage `trenberth_ablation`) dominated the shared
`grad_clip=5.0` L2-norm budget alongside `cloud_albedo` (~400) — three large, partly
anti-correlated gradients competing for one small clip budget, rather than `cloud_albedo`
alone owning it as in the stable 15-param SW-only case. This is what caused the divergence
(best loss 601@batch57 → 1902-2133 by batch 150/300). Validated in `trenberth_gradscale_fix.jl`:
cuts the divergence ratio from 2.97x to 1.25x. Three single-parameter sensitivity sweeps
(`sw_lw_coupling_diagnostic.jl`, `tau0_equator_sensitivity_sweep.jl`, `fl_sensitivity_sweep.jl`)
confirm the underlying SW↔LW coupling is real physics (e.g. d(OLR)/d(cloud_albedo)=-76.5 W/m²
per unit albedo — fixing OSR via cloud reflectivity unavoidably cools the system and pulls
OLR/LRD/LRU down too), not a training artifact — so this `grad_scale` fixes the *instability*,
not the underlying multi-parameter trade-off, which needs further work (see Section 10).

**`emissivity_ocean`/`emissivity_land` get `grad_scale=(0.2, 0.5)`.**
A lower-confidence, reasoned addition (not independently ablation-tested the way tau0/fl were)
— `emissivity_ocean`'s raw gradient was ~75 in a run that included it, large enough to
plausibly cause the same `grad_clip`-budget competition.

**Parameters with zero gradient (excluded):**
- `conv_time_scale` — integer conversion in `Second(time_scale).value` is opaque to Enzyme
- `lsc_rh_threshold` — step-function boundary; gradient is zero almost everywhere
- `infiltration_fraction`, `ocean_moisture`, `snow_melting_threshold` — confirmed **bit-exact**
  zero gradient (not just small) in every run that included them; removed 2026-07-30 rather than
  rescaled. `ocean_moisture` is structural: only read in SpeedyWeather's `initialize!`, before
  the online single-timestep method's differentiated `timestep!` window starts, so no AD fix
  under this training method can help. `infiltration_fraction`/`snow_melting_threshold` both
  route through the same `launch!`-packed `NamedTuple` kernel pattern implicated in
  `lsc_rh_threshold`'s zero gradient above — likely an Enzyme activity-propagation gap through
  that abstraction. (`snow_melting_threshold` previously carried `grad_scale=0.01` here under
  the mistaken assumption its gradient was merely weak, not exactly zero — scaling zero has no
  effect either way.)

In [ ]:
# FIX (2026-07-30, see project_trenberth_lw_transmissivity_gradscale_fix memory):
# this originally diverged (best loss 601@batch57 -> 1902-2133 by batch 150/300,
# see Section 10 ablation below). Root-caused via a 6-stage parameter-count
# ablation + three single-parameter sensitivity sweeps (cloud_albedo,
# tau0_equator, fl; results in examples/output/{sw_lw_coupling_diagnostic,
# tau0_equator_sensitivity_sweep,fl_sensitivity_sweep}/results.csv). Two fixes:
# (1) tau0_equator/tau0_pole/fl never had a grad_scale (unlike
#     absorptivity_water_vapor, which needed one for the same reason: different
#     physical units than the mostly-fractional SW block). Their raw AD
#     gradients (~400/~60/~600) dominated the shared grad_clip=5.0 budget
#     alongside cloud_albedo, fighting a real physical tug-of-war against the
#     SW-albedo block (d(olr)/d(cloud_albedo)=-76.5 etc: fixing OSR via
#     cloud_albedo unavoidably cools the system and drags OLR/LRD/LRU down
#     too -- not a training artifact). grad_scale=(0.04,0.25,0.025) validated
#     in examples/trenberth_gradscale_fix.jl: divergence ratio 2.97x -> 1.25x.
#     emissivity_ocean/land grad_scale is a reasoned (not independently
#     ablation-tested) addition -- emissivity_ocean's raw gradient was ~75 in
#     a run that included it, large enough to plausibly cause the same issue.
# (2) infiltration_fraction/ocean_moisture/snow_melting_threshold had
#     bit-exact-zero AD gradient in every run and are REMOVED, not rescaled.
#     ocean_moisture is structural (only read in `initialize!`, before the
#     online method's differentiated window starts). The other two route
#     through the same `launch!`-packed NamedTuple kernel pattern already
#     known to zero out `lsc_rh_threshold` (MEMORY.md "Parameter AD
#     Limitations") -- likely an Enzyme activity-propagation gap, not
#     something fixable from SpeedyCalibration.jl. Now 20 trainable params.

param_specs = [
    # ── SW cloud reflection ───────────────────────────────────────────────────
    ParamSpec(:cloud_albedo,
        [:shortwave_radiation, :clouds, :cloud_albedo];
        bounds=(0.25f0, 0.95f0), initial=0.60f0),
    ParamSpec(:stratocumulus_cover_max,
        [:shortwave_radiation, :clouds, :stratocumulus_cover_max];
        bounds=(0.25f0, 0.95f0), initial=0.60f0),
    ParamSpec(:stratocumulus_albedo,
        [:shortwave_radiation, :clouds, :stratocumulus_albedo];
        bounds=(0.10f0, 0.90f0), initial=0.50f0),
    ParamSpec(:precipitation_weight,
        [:shortwave_radiation, :clouds, :precipitation_weight];
        bounds=(0.0f0, 0.8f0), initial=0.20f0),

    # ── SW atmospheric absorption ─────────────────────────────────────────────
    ParamSpec(:absorptivity_water_vapor,
        [:shortwave_radiation, :transmissivity, :absorptivity_water_vapor];
        bounds=(60f0, 140f0), initial=75f0, grad_scale=0.01f0),
    ParamSpec(:absorptivity_dry_air,
        [:shortwave_radiation, :transmissivity, :absorptivity_dry_air];
        bounds=(0.005f0, 0.060f0), initial=0.03135f0),
    ParamSpec(:absorptivity_aerosol,
        [:shortwave_radiation, :transmissivity, :absorptivity_aerosol];
        bounds=(0.005f0, 0.060f0), initial=0.03135f0),
    ParamSpec(:ozone_absorption,
        [:shortwave_radiation, :radiative_transfer, :ozone_absorption];
        bounds=(0.002f0, 0.020f0), initial=0.01f0),

    # ── Surface albedo ────────────────────────────────────────────────────────
    ParamSpec(:albedo_land,
        [:albedo, :land, :albedo_land];
        bounds=(0.10f0, 0.70f0), initial=0.40f0),
    ParamSpec(:albedo_high_vegetation,
        [:albedo, :land, :albedo_high_vegetation];
        bounds=(0.04f0, 0.26f0), initial=0.15f0),
    ParamSpec(:albedo_low_vegetation,
        [:albedo, :land, :albedo_low_vegetation];
        bounds=(0.05f0, 0.35f0), initial=0.20f0),
    ParamSpec(:albedo_snow,
        [:albedo, :land, :albedo_snow];
        bounds=(0.15f0, 0.75f0), initial=0.40f0),
    ParamSpec(:snow_depth_scale,
        [:albedo, :land, :snow_depth_scale];
        bounds=(0.005f0, 0.20f0), initial=0.05f0),
    ParamSpec(:albedo_ocean,
        [:albedo, :ocean, :albedo_ocean];
        bounds=(0.02f0, 0.10f0), initial=0.06f0),
    ParamSpec(:albedo_ice,
        [:albedo, :ocean, :albedo_ice];
        bounds=(0.30f0, 0.90f0), initial=0.60f0),

    # ── Longwave transmissivity (Frierson scheme) ─────────────────────────────
    # τ₀_equator and τ₀_pole control optical depth; fₗ is the LW fraction.
    # grad_scale validated in examples/trenberth_gradscale_fix.jl.
    ParamSpec(:tau0_equator,
        [:longwave_radiation, :transmissivity, :τ₀_equator];
        bounds=(2f0, 12f0), initial=6f0, grad_scale=0.04f0),
    ParamSpec(:tau0_pole,
        [:longwave_radiation, :transmissivity, :τ₀_pole];
        bounds=(0.3f0, 4f0), initial=1.5f0, grad_scale=0.25f0),
    ParamSpec(:fl,
        [:longwave_radiation, :transmissivity, :fₗ];
        bounds=(0.0f0, 0.5f0), initial=0.1f0, grad_scale=0.025f0),

    # ── Longwave emissivity ───────────────────────────────────────────────────
    # grad_scale is a reasoned (not independently ablation-tested) addition.
    ParamSpec(:emissivity_ocean,
        [:longwave_radiation, :radiative_transfer, :emissivity_ocean];
        bounds=(0.80f0, 1.00f0), initial=0.98f0, grad_scale=0.2f0),
    ParamSpec(:emissivity_land,
        [:longwave_radiation, :radiative_transfer, :emissivity_land];
        bounds=(0.80f0, 1.00f0), initial=0.98f0, grad_scale=0.5f0),

    # ── Land hydrology params REMOVED ─────────────────────────────────────────
    # infiltration_fraction, ocean_moisture, snow_melting_threshold all had
    # bit-exact-zero AD gradient in every run (dead weight under the online
    # single-timestep training method) -- see fix note above.
]

println("$(length(param_specs)) trainable parameters defined.")

## 3. Configure the Loss Function

`TRENBERTH_LOSS` targets all 6 SW+LW fluxes with `osr`/`olr` at full weight (1.0) and surface fluxes at 0.3-0.5, reflecting their larger observational uncertainty — except `lrd`, whose weight was corrected from 0.3 to **0.7** on 2026-08-01.

**Why 0.7, not the package default 0.3:** an initial reweighting test (0.3→1.0) looked like it made things worse — but that compared each run's *self-reported* loss, computed under a *different* weight function each time, which is not a valid comparison (a larger `lrd` weight mechanically inflates its own contribution to the total even if its bias improves). Rescoring every candidate's actual raw flux biases under one **fixed** reference weighting (the original 0.3-weighted `TRENBERTH_LOSS`) gave a completely different, corrected picture: a finer sweep (0.3/0.4/0.5/0.6/0.7/0.8/1.0) found a clean local optimum at **weight=0.7** (fixed-yardstick score 96.0, vs. 153.7 at the original 0.3 — a 38% reduction), confirmed to hold at true 7-year climate equilibrium too (score 369.6 vs. 535.6 — a 31% reduction), not just the training proxy. See `project_trenberth_lw_transmissivity_gradscale_fix` memory, "CORRECTION 2026-08-01" section, for the full derivation.

In [ ]:
# lrd weight corrected 0.3 -> 0.7 (2026-08-01) -- see the markdown note above and
# project_trenberth_lw_transmissivity_gradscale_fix memory for the full derivation.
# Everything else identical to the package's TRENBERTH_LOSS.
loss_config = LossConfig(
    [:osr, :sru, :srd, :olr, :lrd, :lru];
    targets = Dict(:osr => 101.9f0, :sru =>  23.0f0, :srd => 168.0f0,
                   :olr => 235.0f0, :lrd => 333.0f0, :lru => 398.0f0),
    weights = Dict(:osr => 1.0f0,   :sru =>  0.5f0,  :srd => 0.5f0,
                   :olr => 1.0f0,   :lrd =>  0.7f0,  :lru => 0.3f0),
)

println("Loss configuration: $(length(loss_config.flux_keys))-flux MSE")
println()
@printf("  %-6s  %8s  %6s\n", "flux", "target", "weight")
println("  " * "-" ^ 24)
for k in loss_config.flux_keys
    @printf("  %-6s  %6.1f W/m²  %.1f\n",
            k, loss_config.targets[k], loss_config.weights[k])
end

## 4. Quick Test

Verify all parameter paths are correct and Enzyme can differentiate through everything before the full run.

> **Note:** `calibrate!` warms up Enzyme automatically on the actual training model (`warmup_enzyme=true` by default). Expect *"Enzyme warmup complete in X s."* before the spinup on the first call. Pass `warmup_enzyme=false` on subsequent calls in the same session.

In [ ]:
# dt=Minute(20): the auto-scaled Δt at trunc=5 works out to ~3h (vs. 40min at the
# full run's trunc=31), which is too large to stay numerically stable with the full
# physics package -- it was hitting a NaN/Inf blowup by time step 10, and everything
# downstream of that (all 20 gradient-sample attempts in batch 1) then crawled through
# corrupted/subnormal floating-point state for ~1 hour before finally giving up.
# Overriding to a smaller, known-stable Δt (same value already used for climate
# validation runs elsewhere in this notebook) keeps the quick test a fast sanity check.
result_test = calibrate!(
    param_specs,
    Optimisers.Adam(1f-2),
    loss_config,
    quick_test_config(dt=Minute(20)),
)

println("Quick test complete: ", result_test.conv_info.stop_reason)

# Check gradient magnitudes — a zero gradient for any parameter is a red flag
println("\nGradient magnitudes (last batch):")
@printf("  %-28s  %12s\n", "parameter", "|mean grad|")
println("  " * "-" ^ 45)
for spec in result_test.param_specs
    g = result_test.history[Symbol("grad_", spec.name)]
    isempty(g) && continue
    @printf("  %-28s  %12.3e\n", spec.name, abs(g[end]))
end

**Interpreting gradient magnitudes:**  
- All gradients should be non-zero. A zero gradient means the parameter is disconnected from the loss through the AD graph — check the path or exclude the parameter.
- If one gradient is 100× larger than the others, consider increasing `grad_scale` for the weaker ones, or reducing it for the outlier.

---

## 5. Full Training Run

**⏱ Expected runtime: ~8–12 hours at T31.**

We use a slightly lower initial LR (`5f-3`) compared to the SW-only run because the 6-flux loss is more sensitive and can oscillate with a high LR.

**Status (2026-08-01): the calibration is now well-understood and validated end-to-end.** Full story, chronologically:

**2026-07-30**: the grad_scale fix stopped the catastrophic blowup, but training still didn't *converge* — `batch_days=10` found its best point at batch 60 (889.4), then climbed steadily for the rest of the run through 3 LR decays with no recovery.

**2026-07-31**: root-caused the drift. It wasn't post-optimum at all — `lrd`'s bias grew monotonically from batch 1 (masked early by `srd`'s much larger, faster-shrinking initial bias), and key gradients never once flipped sign across 240 batches — a persistent, low-noise signal, not oscillation. A `batch_days` sensitivity sweep (30→10→5→3→2) found shorter windows are strictly better on every metric: `batch_days=2` reaches a true floor of `best_smoothed_loss=148.8` (vs. 889.4), with 5 of 6 fluxes essentially nailed and only `lrd` (+21 W/m²) left as an open gap. This was adopted as the production config and validated: real climate-equilibrium data confirmed `batch_days=2` is genuinely far better than `batch_days=10`, not just on the training proxy — though `batch_days=10` is actually *better* on `srd`/`lrd` specifically at equilibrium, it just loses overall from being so much worse on `osr`/`olr`.

**2026-08-01**: an initial attempt to fix the remaining `lrd` gap via loss-reweighting (0.3→1.0) looked like a dead end — but that comparison was invalid (it compared self-reported losses computed under *different* weight functions, not an apples-to-apples comparison). Rescoring under one fixed reference weighting revealed reweighting *does* work, and a finer sweep (0.3 through 1.0 in steps of 0.1) found a clean local optimum at `lrd` weight=**0.7**: fixed-yardstick score 96.0 vs. 153.7 at weight 0.3 (38% better) on the training proxy, and — critically, confirmed at true 7-year climate equilibrium — score 369.6 vs. 535.6 (31% better), not just a training-metric artifact.

**Final production config (below): `batch_days=2`, `samples_per_batch=10`, `lrd` weight=**0.7**.** At true equilibrium this reaches `osr` +1.3, `sru` -3.0, `srd` -10.4, `olr` +13.6, `lrd` +20.5, `lru` +1.1 W/m² bias — `osr`/`sru`/`lru` excellent, `srd`/`olr`/`lrd` improved substantially over the untrained default (-29.1/+37.5/+8.8 respectively) though not fully closed. **Practical takeaway:** always use `result.best_params`, never `result.final_params` — training still drifts past its optimum eventually, patience just catches it well past a much better point than the original `batch_days=10` config ever reached. Full derivation, every intermediate result, and the retracted-then-corrected reweighting conclusion are in `project_trenberth_lw_transmissivity_gradscale_fix` memory.

In [ ]:
# HP choice (2026-07-31, lrd weight corrected 2026-08-01): a full batch_days
# sensitivity sweep (30/10/5/3/2, each with samples_per_batch chosen so
# steps_per_sample keeps gcd(steps_per_sample, steps_per_day=36)=1 -- avoids the
# diurnal-aliasing failure mode described in Section 2's fix note) found
# batch_days is the dominant lever for this 20-param/6-flux problem, and shorter
# is strictly better on every metric tested:
#
#   batch_days     grace period    best_smoothed_loss
#      30               0              1126.0
#      10              ~60              889.4    (original config)
#       5              106               551.9
#       3              131               348.8
#       2              331 (true floor)  148.8    <- adopted (at lrd weight=0.3)
#
# Mechanism (see project_trenberth_lw_transmissivity_gradscale_fix memory for
# full derivation): this is one continuously-run simulation with no state
# reset between batches, so a *longer* single batch window gives the slow LW/
# thermal feedback (cloud_albedo/fl/tau0_equator -> lrd, confirmed real
# physics in sw_lw_coupling_diagnostic.jl) more time to compound *within* that
# one batch's own averaging window, biasing the batch-mean gradient sooner
# rather than later.
#
# At batch_days=2 (lrd weight=0.3), 5 of 6 fluxes were essentially nailed but
# lrd alone (bias +21.2) accounted for ~89% of the remaining loss. See the
# Section 3 markdown for the full lrd-reweighting story (2026-08-01): a finer
# sweep (0.3 through 1.0) found a clean local optimum at lrd weight=0.7 --
# fixed-yardstick score 96.0 vs. 153.7 at weight 0.3 on the training proxy
# (38% better), and 369.6 vs. 535.6 at true 7-year climate equilibrium (31%
# better) -- loss_config (Section 3, above) already reflects this.
#
# samples_per_batch=10 at batch_days=2: batch_steps=ceil(2*36)=72,
# steps_per_sample=72÷10=7, gcd(7,36)=1 -- clean, full diurnal coverage.
#
# max_batches=400, enable_lr_decay=false: matches the exact validated
# sensitivity-test config (examples/trenberth_bd2_lrdweight07.jl) that
# produced the numbers above -- patience (default 30) is the real stopping
# criterion. loss_threshold disabled (1f-6): the old 500/25 values were tuned
# for different (batch_days=10) loss floors and are irrelevant at this floor.
#
# Resume-safe (matching Section 10's ablation pattern): if
# output/trenberth_full_result.jld2 already exists, load it instead of
# re-training. This exact config (batch_days=2, samples_per_batch=10,
# lrd weight=0.7) was already run standalone in
# examples/trenberth_bd2_lrdweight07.jl (identical param_specs, loss_config,
# and TrainingConfig, verified byte-for-byte) -- no need to duplicate ~50
# minutes of compute if that result is already on disk.
save_path = joinpath(@__DIR__, "output", "trenberth_full_result.jld2")
if isfile(save_path)
    result = load_result(save_path)
    println("Loaded existing result from: $save_path")
    println("  best_batch: ", result.conv_info.best_batch,
            "  best_smoothed_loss: ", round(result.conv_info.best_smoothed_loss, digits=2),
            "  stop_reason: ", result.conv_info.stop_reason)
else
    result = calibrate!(
        param_specs,
        Optimisers.Adam(5f-3),
        loss_config,
        TrainingConfig(
            spinup_days       = 180,
            batch_days        = 2.0,
            samples_per_batch = 10,
            max_batches       = 400,
            loss_threshold    = 1f-6,
            enable_lr_decay   = false,
            trunc             = 31,
            nlayers            = 8,
            daily_cycle       = true,   # required: physical diurnal cycle must stay on
        ),
    )
    mkpath(dirname(save_path))
    save_result(result, save_path)
    println("Result saved to: $save_path")
end

## 6. Inspect Results

In [ ]:
println(result)
println()
println("Convergence info:")
println("  stop_reason:        ", result.conv_info.stop_reason)
println("  best_smoothed_loss: ", round(result.conv_info.best_smoothed_loss, digits=2))
println("  total_batches:      ", result.conv_info.total_batches)
@printf("  total_time:         %.1f hours\n", result.conv_info.total_time / 3600)

In [ ]:
# Parameter table: initial → best checkpoint (result.best_params, not final_params --
# see the Section 5 status note: training can drift past its optimum, so the
# lowest-smoothed-loss checkpoint is the representative one, not the last batch trained)
@printf("\n%-28s  %10s  %10s  %10s\n", "parameter", "initial", "best", "change")
println("-" ^ 65)
for spec in result.param_specs
    init    = isnothing(spec.initial) ? NaN32 : spec.initial
    trained = result.best_params[spec.name]
    @printf("%-28s  %10.4g  %10.4g  %+10.4g\n",
            spec.name, init, trained, trained - init)
end

In [ ]:
# Flux bias at the best checkpoint (result.conv_info.best_batch, not the last batch --
# same best_params vs final_params reasoning as the parameter table above)
@printf("\n%-6s  %8s  %8s  %8s\n", "flux", "target", "best", "bias")
println("-" ^ 38)
for k in result.loss_config.flux_keys
    tgt  = result.loss_config.targets[k]
    val  = result.history[k][result.conv_info.best_batch]
    @printf("%-6s  %8.2f  %8.2f  %+8.2f\n", k, tgt, val, val - tgt)
end

## 7. Plot Training History

In [ ]:
figs = plot_training(result; save_dir=joinpath(@__DIR__, "output", "trenberth_full"))
figs.fig_loss

In [ ]:
figs.fig_flux

In [ ]:
figs.fig_params

In [ ]:
figs.fig_grads

## 8. Climate Validation

We compare the default model's equilibrium climatology against the trained one across all 6 target fluxes, precipitation, and the temperature profile.

**⏱ Expected runtime: ~1–3 hours at T31.**

In [ ]:
clm = run_climate_validation(result; n_years=7, stat_years=5, dt=Minute(20))

In [ ]:
# Full bias comparison table
targets = result.loss_config.targets

@printf("%-6s  %8s  %9s  %9s  %9s  %9s\n",
        "flux", "target",
        "def val", "def bias",
        "trn val", "trn bias")
println("-" ^ 60)
for k in result.loss_config.flux_keys
    tgt    = targets[k]
    d_val  = getproperty(clm.default, k)
    t_val  = getproperty(clm.trained, k)
    @printf("%-6s  %8.2f  %9.2f  %+9.2f  %9.2f  %+9.2f\n",
            k, tgt, d_val, d_val-tgt, t_val, t_val-tgt)
end
println()

# Also show precipitation (not in the loss — a held-out diagnostic)
println("Held-out diagnostics (not in loss):")
@printf("  Precipitation:  default = %.2f mm/day  trained = %.2f mm/day  (ERA5 ≈ 2.74)\n",
        clm.default.precip_total, clm.trained.precip_total)

In [ ]:
cfigs = plot_climate(clm;
    save_dir    = joinpath(@__DIR__, "output", "trenberth_full"),
    loss_config = result.loss_config,
)
cfigs.fig_rad

In [ ]:
cfigs.fig_lw

In [ ]:
cfigs.fig_precip

In [ ]:
cfigs.fig_summary

## 9. Interpretation and Next Steps

**Reading the summary plot:**  
Each bar shows the equilibrium bias (trained value − Trenberth target) for one flux. Bars shrinking toward zero from default (grey) to trained (blue) indicate successful calibration. Watch for biases that *grow* in fluxes not in the loss — these indicate compensatory parameter adjustments.

**Where this calibration currently stands (2026-08-01):** at true 7-year climate equilibrium, `osr`/`sru`/`lru` are excellent (within a few W/m² of target), `srd`/`olr` are substantially improved over the untrained default but not fully closed, and `lrd` remains the largest residual (~+20 W/m², down from ~+30 before the `lrd`-weight correction, but still the dominant error term). This is a genuine, validated, partial success — not a fully converged calibration on all 6 fluxes.

**Common issues and fixes:**

| Symptom | Likely cause | Fix |
|---------|-------------|-----|
| Loss oscillates without decreasing | LR too high | Reduce Adam LR to `1f-3` |
| One flux improves, another degrades | Loss weights too uneven | Rebalance `loss_config` weights (see Section 3 for how the current `lrd=0.7` was found — always rescore candidates under one *fixed* reference weighting, never compare self-reported losses across different weight configs) |
| Parameter hits its bound | Bounds too narrow | Widen the offending `ParamSpec` bounds |
| Precipitation degrades strongly | Convective params pulled too far | Add precipitation term to loss |
| Gradient ~0 for a parameter | Zero-gradient path | Exclude parameter or change its path |

**Closing the remaining `lrd` gap — not yet attempted:** staged/curriculum training. Converge the SW block alone first (the already-working 15-param SW-only config converges cleanly), freeze those parameters, train *only* the LW-block params (`tau0_equator`, `tau0_pole`, `fl`, `emissivity_ocean`, `emissivity_land`) against `olr`/`lrd`/`lru` from that already-good starting point, then a short joint fine-tune. Motivation: loss-reweighting can only move along an existing SW/LW trade-off curve (which is what Section 3's `lrd`-weight correction did); staged training might change what's reachable at all, rather than just picking a different point on the same curve. See `project_trenberth_lw_transmissivity_gradscale_fix` memory for the full reasoning.

**Continuing from this result:**  
Set `initial = result.best_params[spec.name]` (not `final_params` — see Section 5/6) in each `ParamSpec` and re-run with a lower LR to refine further. The sigmoid reparameterisation ensures bounds are still respected.

## 10. Diagnosing Instability: Parameter-Count Ablation

**Motivation:** the original `loss_threshold = 500` (Section 5) is suspected to have existed
because training the full 23-parameter set for longer risks the loss *exploding* rather than
continuing to converge. Now that the threshold is lowered to 25 (Section 5) so training runs
much further, we need to check whether that suspicion holds — and if so, *where in parameter
space* the instability originates.

**Method:** train nested, growing subsets of `param_specs`, in the physical-module order the
list is already organised in:

| Stage | Adds | Cumulative n |
|-------|------|--------------|
| 1 | SW clouds (`cloud_albedo`, `stratocumulus_cover_max`, `stratocumulus_albedo`, `precipitation_weight`) | 4 |
| 2 | + SW absorption (`absorptivity_water_vapor`, `absorptivity_dry_air`, `absorptivity_aerosol`, `ozone_absorption`) | 8 |
| 3 | + Surface albedo (7 params: land/vegetation/snow/ocean/ice) | 15 |
| 4 | + LW transmissivity (`tau0_equator`, `tau0_pole`, `fl`) | 18 |
| 5 | + LW emissivity (`emissivity_ocean`, `emissivity_land`) | 20 |
| 6 | + Land hydrology (`infiltration_fraction`, `ocean_moisture`, `snow_melting_threshold`) | 23 (full) |

Each stage trains only its first `n` parameters from `param_specs`; params beyond the cutoff
stay at model defaults (not at any other stage's trained value) so stages are independent,
clean ablations. `loss_threshold` is set to `1f-6` (never triggers) so every stage always runs
its full 150-batch budget — we want to see the *entire* trajectory, including any late blow-up,
not stop early. All other hyperparameters match Section 5's full run.

Results are cached to `output/trenberth_ablation/<stage>/result.jld2` — resume-safe; re-running
this cell after an interruption skips completed stages.

In [ ]:
struct AblationStage
    name     :: String
    label    :: String
    n_params :: Int
    color    :: Symbol
end

_ablation_cutoffs = [4, 8, 15, 18, 20, 23]
_ablation_labels  = [
    "SW clouds",
    "+SW absorption",
    "+Surface albedo",
    "+LW transmissivity",
    "+LW emissivity",
    "+Land hydrology (full)",
]
_ablation_colors = [:firebrick, :darkorange, :goldenrod, :seagreen, :steelblue, :purple]

ablation_stages = [
    AblationStage(@sprintf("stage%d_n%02d", i, n), lbl, n, col)
    for (i, (n, lbl, col)) in enumerate(zip(_ablation_cutoffs, _ablation_labels, _ablation_colors))
]

println("Ablation stages:")
for st in ablation_stages
    @printf("  %-24s  %-24s  n=%2d params\n", st.name, st.label, st.n_params)
end

In [ ]:
const ABLATION_DIR = joinpath(@__DIR__, "output", "trenberth_ablation")

function _ablation_cfg()
    TrainingConfig(
        spinup_days       = 180,
        batch_days        = 10.0,
        samples_per_batch = 32,
        max_batches       = 150,
        loss_threshold    = 1f-6,   # disabled: always run the full batch budget
        trunc             = 31,
        nlayers           = 8,
        daily_cycle       = true,   # required: physical diurnal cycle must stay on
    )
end

ablation_results = Dict{String, TrainingResult}()

for st in ablation_stages
    rfile = joinpath(ABLATION_DIR, st.name, "result.jld2")
    if isfile(rfile)
        r = load_result(rfile)
        ablation_results[st.name] = r
        @printf("%-24s  n=%2d  loaded from disk (best=%.1f, final=%.1f, stop=%s)\n",
                st.label, st.n_params, r.conv_info.best_smoothed_loss,
                r.history[:smoothed_loss][end], r.conv_info.stop_reason)
        continue
    end
    @printf("\n%-24s  n=%2d  training...\n", st.label, st.n_params)
    flush(stdout)
    r = calibrate!(
        param_specs[1:st.n_params],
        Optimisers.Adam(1f-2),
        loss_config,
        _ablation_cfg();
        save_dir = joinpath(ABLATION_DIR, st.name),
    )
    ablation_results[st.name] = r
    @printf("  done: %d batches, best=%.1f, final=%.1f, stop=%s\n",
            r.conv_info.total_batches, r.conv_info.best_smoothed_loss,
            r.history[:smoothed_loss][end], r.conv_info.stop_reason)
end

In [ ]:
# Summary table with a simple explosion heuristic: flag a stage if its final
# smoothed loss is >50% above its own minimum (i.e. it got better, then got
# meaningfully worse again within the 150-batch budget).
println("\nAblation summary")
@printf("%-24s  %4s  %8s  %8s  %8s  %s\n",
        "Stage", "n", "Min L", "Best L", "Final L", "Verdict")
println("-" ^ 80)

for st in ablation_stages
    r = get(ablation_results, st.name, nothing)
    isnothing(r) && continue
    nb = r.conv_info.total_batches
    l  = r.history[:smoothed_loss][1:nb]
    lossmin, imin = findmin(l)
    lossfin = l[end]
    exploded = lossfin > 1.5f0 * lossmin
    verdict  = exploded ? "⚠ EXPLODES after batch $imin (min→final: $(round(lossmin,digits=1))→$(round(lossfin,digits=1)))" : "stable"
    @printf("%-24s  %4d  %8.1f  %8.1f  %8.1f  %s\n",
            st.label, st.n_params, lossmin, r.conv_info.best_smoothed_loss, lossfin, verdict)
end

In [ ]:
_ax_abl = (xgridvisible=false, ygridvisible=false,
           topspinevisible=false, rightspinevisible=false)

fig_abl = Figure(size=(900, 500), fontsize=13)
ax_abl = Axis(fig_abl[1,1];
    title="Smoothed loss vs. batch, by parameter-set size",
    xlabel="Batch", ylabel="Smoothed loss", yscale=log10, _ax_abl...)

for st in ablation_stages
    r = get(ablation_results, st.name, nothing)
    isnothing(r) && continue
    nb = r.conv_info.total_batches
    lines!(ax_abl, 1:nb, r.history[:smoothed_loss][1:nb];
           label="$(st.label) (n=$(st.n_params))", color=st.color, linewidth=2)
end
axislegend(ax_abl; position=:rt, framevisible=false, labelsize=10)

save(joinpath(ABLATION_DIR, "ablation_loss_curves.pdf"), fig_abl)
fig_abl

In [ ]:
fig_abl_flux = Figure(size=(1800, 320), fontsize=12)
_flux_keys_abl  = [:osr, :sru, :srd, :olr, :lrd, :lru]
_flux_label_abl = Dict(:osr=>"OSR", :sru=>"SRU", :srd=>"SRD",
                        :olr=>"OLR", :lrd=>"LRD", :lru=>"LRU")

for (col, fk) in enumerate(_flux_keys_abl)
    ax = Axis(fig_abl_flux[1, col];
              title  = _flux_label_abl[fk],
              xlabel = "Batch",
              ylabel = col == 1 ? "W m⁻²" : "",
              _ax_abl...)
    hlines!(ax, [loss_config.targets[fk]]; color=:black, linestyle=:dash,
            linewidth=1.5, label="Target")
    for st in ablation_stages
        r = get(ablation_results, st.name, nothing)
        isnothing(r) && continue
        nb = r.conv_info.total_batches
        lines!(ax, 1:nb, r.history[fk][1:nb];
               color=(st.color, 0.8), linewidth=1.3, label=st.label)
    end
    col == 1 && axislegend(ax; position=:rt, framevisible=false, labelsize=8)
end

Label(fig_abl_flux[0,:],
    "Ablation: per-flux trajectories by parameter-set size (— = Trenberth target)",
    fontsize=14, font=:bold)

save(joinpath(ABLATION_DIR, "ablation_flux_curves.pdf"), fig_abl_flux)
fig_abl_flux